In [ ]:
using Plots
using LinearAlgebra
using Krylov
using Printf
using LaTeXStrings
using BenchmarkTools
include("poisson2d.jl")

In [ ]:
default(lw=2, markersize=6,
    xtickfont=font(12), ytickfont=font(12),
    guidefont=font(14), legendfont=font(12), titlefont=font(12))

# Check Matrix Construction

In [ ]:
Lx = 3;
Ly = 1;
nx = 20; # (nx-1)×(ny-1) interior points
ny = 40;
Nx = nx-1;
Ny = ny-1;

x = LinRange(0, Lx, nx + 1)
y = LinRange(0, Ly, ny + 1)

@show Δx = x[2] - x[1];
@show Δy = y[2] - y[1];

A = assemble_laplacian2d(Δx, Δy, Nx, Ny);
L1 = dirichlet_laplacian_2d_kron(Δx, Δy, Nx, Ny);
L2 = dirichlet_laplacian_2d_kron2(Δx, Δy, Nx, Ny);


In [ ]:
@show norm(L1-A);
@show norm(L2-A);
@show norm(Matrix(L1-A));
@show norm(Matrix(L2-A));

In [ ]:
A

# Test Problem

In [ ]:
Lx = 1;
Ly = 1;
nx = 20; # (nx-1)×(ny-1) interior points
ny = 20;
Nx = nx-1;
Ny = ny-1;

x = LinRange(0, Lx, nx + 1)
y = LinRange(0, Ly, ny + 1)

@show Δx = x[2] - x[1];
@show Δy = y[2] - y[1];

L = dirichlet_laplacian_2d_kron2(Δx, Δy, Nx, Ny);

In [ ]:
xy = [[x_, y_] for x_ in x, y_ in y]; # mesh including boundary points
xy_int = [[x_, y_] for x_ in x[2:end-1],y_ in y[2:end-1]]; # interior mesh points


uex= X -> X[1] *(1-X[1]) * X[2] * (1-X[2]);
f = X -> 2 * X[2] * (1-X[2]) + 2 * X[1] * (1-X[1]);
B = vec(f.(xy_int))
# u = zeros(Nx,Ny);
# U = vec(u);
# U .= (-L\B);
U = -L\B;
u = reshape(U, Nx, Ny);


In [ ]:
p1 = contourf(x[2:end-1], y[2:end-1], u', title="Finite Difference Solution", xlabel="x", ylabel="y", clims=(0, 0.065));
xlims!(p1, 0, 1);
ylims!(p1, 0, 1);
# clims!(p1, 0, 0.065);
p2 = contourf(x, y, uex.(xy)', title="Exact Solution", xlabel="x", ylabel="", clims=(0, 0.065));
# p2 = contourf(x[2:end-1], y[2:end-1], uex.(xy_int)', title="Exact Solution", xlabel="x", ylabel="", clims=(0, 0.065));
xlims!(p2, 0, 1);
ylims!(p2, 0, 1);
# clims!(p2, 0, 0.065);
plot(p1, p2, layout=(1,2), size=(1000,400))
savefig("poisson2d_example.pdf")


In [ ]:
@show norm(u-uex.(xy_int))

# Trig Examples

In [ ]:
Lx = 1;
Ly = 1;
nx = 50; # (nx-1)×(ny-1) interior points
ny = 50;
Nx = nx-1;
Ny = ny-1;

x = LinRange(0, Lx, nx + 1)
y = LinRange(0, Ly, ny + 1)

@show Δx = x[2] - x[1];
@show Δy = y[2] - y[1];

L = dirichlet_laplacian_2d_kron2(Δx, Δy, Nx, Ny);

xy = [[x_, y_] for x_ in x, y_ in y]; # mesh including boundary points
xy_int = [[x_, y_] for x_ in x[2:end-1],y_ in y[2:end-1]]; # interior mesh points


In [ ]:


kx = 3;
ky = 2;
# X = (x,y) in 2D
uex = X -> sin(kx * π * X[1] / Lx) * sin(ky * π * X[2] / Ly);
f = X -> ((kx * π / Lx)^2 + (ky * π / Ly)^2) * uex(X);


B = vec(f.(xy_int))
U = -L\B;
u = reshape(U, Nx, Ny);


In [ ]:
p1 = contourf(x[2:end-1], y[2:end-1], u', title="Finite Difference Solution", xlabel="x", ylabel="y", clims=(-1,1));
xlims!(p1, 0, 1);
ylims!(p1, 0, 1);
# clims!(p1, 0, 0.065);
p2 = contourf(x, y, uex.(xy)', title="Exact Solution", xlabel="x", ylabel="", clims=(-1,1));
# p2 = contourf(x[2:end-1], y[2:end-1], uex.(xy_int)', title="Exact Solution", xlabel="x", ylabel="", clims=(0, 0.065));
xlims!(p2, 0, 1);
ylims!(p2, 0, 1);
# clims!(p2, 0, 0.065);
plot(p1, p2, layout=(1,2), size=(1000,400))
# savefig("poisson2d_example.pdf")


In [ ]:
@show norm(u-uex.(xy_int))

In [ ]:
@which A\B

# Timing

## Sparse Matrices

In [ ]:
n_vals = [5, 10, 20, 40, 80, 160, 320, 640, 1280, 2560, 5120];
N_vals = n_vals.^2;
time_vals = [];
kx = 1; ky = 1;
Lx = 1; Ly = 1;
for n in n_vals
    nx = n; # (nx-1)×(ny-1) interior points
    ny = n;
    Nx = nx-1;
    Ny = ny-1;

    uex = X -> sin(kx * π * X[1] / Lx) * sin(ky * π * X[2] / Ly);
    f = X -> ((kx * π / Lx)^2 + (ky * π / Ly)^2) * uex(X);

    x = LinRange(0, 1, nx + 1)
    y = LinRange(0, 1, ny + 1)
    Δx = x[2] - x[1];
    Δy = y[2] - y[1];
    
    L = dirichlet_laplacian_2d_kron2(Δx, Δy, Nx, Ny);
    A = -L;

    xy_int = [[x_, y_] for x_ in x[2:end-1],y_ in y[2:end-1]]; # interior mesh points

    B = vec(f.(xy_int))
    stats = @btimed $A \ $B;
    push!(time_vals, stats.time);
    println("n = $n, N = $(n^2), time = $(stats.time) seconds");
end

In [ ]:
scatter(N_vals, time_vals, xscale=:log10, yscale=:log10, 
    xlabel="N", ylabel="Time (seconds)", title="Time to Solve Poisson in 2D",label="", legend=:bottomright)
plot!(N_vals, 1e-6* N_vals, label="O(N)", linestyle=:dash)
plot!(N_vals, 1e-8* N_vals.^(3/2), label="O(N^{3/2})", linestyle=:dash)

## Dense Matrices

In [ ]:
n_vals = [5, 10, 20, 40, 80, 160];
N_vals = n_vals.^2;
Lx = 1; Ly = 1;
kx = 1; ky = 1;
time_vals = [];
for n in n_vals
    nx = n; # (nx-1)×(ny-1) interior points
    ny = n;
    Nx = nx-1;
    Ny = ny-1;

    uex = X -> sin(kx * π * X[1] / Lx) * sin(ky * π * X[2] / Ly);
    f = X -> ((kx * π / Lx)^2 + (ky * π / Ly)^2) * uex(X);

    x = LinRange(0, 1, nx + 1)
    y = LinRange(0, 1, ny + 1)
    Δx = x[2] - x[1];
    Δy = y[2] - y[1];
    
    L = dirichlet_laplacian_2d_kron2(Δx, Δy, Nx, Ny);
    A = Matrix(-L);

    xy_int = [[x_, y_] for x_ in x[2:end-1],y_ in y[2:end-1]]; # interior mesh points

    B = vec(f.(xy_int))
    # @btime $A \ $B;
    stats = @btimed $A \ $B;
    push!(time_vals, stats.time);
    println("n = $n, N = $(n^2), time = $(stats.time) seconds");

end

In [ ]:
scatter(N_vals, time_vals, xscale=:log10, yscale=:log10, 
    xlabel="N", ylabel="Time (seconds)", title="Time to Solve Poisson in 2D",label="", legend=:bottomright)
plot!(N_vals, 1e-8* N_vals, label="O(N)", linestyle=:dash)
plot!(N_vals, 1e-12* N_vals.^(3), label="O(N^{3})", linestyle=:dash)

## Krylov

In [ ]:
using Krylov

In [ ]:
n_vals = [5, 10, 20, 40, 80, 160, 320, 640, 1280, 2560];
kx = 1; ky = 1;
Lx = 1; Ly = 1;
time_vals = [];

for n in n_vals
    nx = n; # (nx-1)×(ny-1) interior points
    ny = n;
    Nx = nx-1;
    Ny = ny-1;

    uex = X -> sin(kx * π * X[1] / Lx) * sin(ky * π * X[2] / Ly);
    f = X -> ((kx * π / Lx)^2 + (ky * π / Ly)^2) * uex(X);

    x = LinRange(0, 1, nx + 1)
    y = LinRange(0, 1, ny + 1)
    Δx = x[2] - x[1];
    Δy = y[2] - y[1];
    
    L = dirichlet_laplacian_2d_kron2(Δx, Δy, Nx, Ny);
    A = -L;

    xy_int = [[x_, y_] for x_ in x[2:end-1],y_ in y[2:end-1]]; # interior mesh points

    B = vec(f.(xy_int))
    
    # @btime $A \ $B;
    # @btime cg($A, $B);
    stats = @btimed cg($A, $B);
    push!(time_vals, stats.time);
    println("n = $n, N = $(n^2), time = $(stats.time) seconds");
end

In [ ]:
scatter(N_vals, time_vals, xscale=:log10, yscale=:log10, 
    xlabel="N", ylabel="Time (seconds)", title="Time to Solve Poisson in 2D",label="", legend=:bottomright)
plot!(N_vals, 1e-8* N_vals, label="O(N)", linestyle=:dash)
# plot!(N_vals, 1e-12* N_vals.^(3), label="O(N^{3})", linestyle=:dash)

In [ ]:
n_vals = [5, 10, 20, 40, 80, 160, 320, 640, 1280, 2560, 5120];
kx = 1; ky = 1;
for n in n_vals
    nx = n; # (nx-1)×(ny-1) interior points
    ny = n;
    Nx = nx-1;
    Ny = ny-1;

    uex = X -> sin(kx * π * X[1] / Lx) * sin(ky * π * X[2] / Ly);
    f = X -> ((kx * π / Lx)^2 + (ky * π / Ly)^2) * uex(X);

    x = LinRange(0, 1, nx + 1)
    y = LinRange(0, 1, ny + 1)
    Δx = x[2] - x[1];
    Δy = y[2] - y[1];
    
    L = dirichlet_laplacian_2d_kron2(Δx, Δy, Nx, Ny);
    A = -L;

    xy_int = [[x_, y_] for x_ in x[2:end-1],y_ in y[2:end-1]]; # interior mesh points

    B = vec(f.(xy_int))

    U = cg(A, B)[1];
    u = reshape(U, Nx, Ny);
    @show norm(u-uex.(xy_int),Inf);

end